# Description

In this notebook, we will prepare the datasets for the concept discovery in the VML experiment. The following datasets will be utilized:

1. **ImageNet**: A large-scale dataset for image classification and object detection tasks.
2. **MSCOCO**: A dataset designed for object detection, segmentation, and captioning tasks.
3. **CIFAR-10**: A smaller dataset consisting of 10 classes commonly used for image classification tasks.

The preparation steps will include downloading, preprocessing, and organizing the datasets for further analysis.

### Description
This cell prepares the ImageNet dataset by organizing it into a structured directory format suitable for training and validation experiments. The dataset is divided into `train` and `val` directories, with subfolders named after class labels. Each subfolder contains the corresponding images for that class.

### Steps: IMAGENET
1. **Import Required Modules**: Import `os` for directory operations and `shutil` for file copying.
2. **Define Paths**: Specify the paths for the downloaded ImageNet data and the directory where the prepared data will be stored.
3. **Create Directories**: Create `train` and `val` directories in the prepared data directory.
4. **Define a Function**: Write a function to create subfolders for each class and populate them with the corresponding images.
5. **Organize Data**:
    - For the training data, create subfolders for each class in the `train` directory and copy the images.
    - For the validation data, create subfolders for each class in the `val` directory and copy the images.
6. **Completion Message**: Print a message indicating that the ImageNet data preparation is complete.


In [11]:
download_data_raw_dir = "/mnt/abka03/adam/adam_data/rawdata/imagenet"
imagenet_url = "https://image-net.org/data/winter21_whole.tar.gz"

In [12]:
import requests
import os

def download_imagenet_data(url, save_dir):
    """
    Downloads a file from the given URL and saves it to the specified directory.

    Parameters:
    - url (str): The URL of the file to download.
    - save_dir (str): The directory where the file will be saved.

    Returns:
    - str: The path to the downloaded file.
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    elif os.listdir(save_dir):
        print(f"Directory {save_dir} is not empty. Using existing folder.")
        return None

    local_filename = os.path.join(save_dir, url.split('/')[-1])
    with requests.get(url, stream=True) as response:
        response.raise_for_status()
        with open(local_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    print(f"Downloaded file to {local_filename}")
    return local_filename

In [13]:
download_imagenet_data(imagenet_url, download_data_raw_dir)

In [4]:
import os
import tarfile
import shutil

def safe_extract(tar, path=".", members=None, *, numeric_owner=False):
    """
    Safely extract tar files to prevent path traversal attacks.
    """
    for member in tar.getmembers():
        member_path = os.path.join(path, member.name)
        abs_path = os.path.abspath(member_path)
        if not abs_path.startswith(os.path.abspath(path)):
            raise Exception("Attempted Path Traversal in Tar File")
    tar.extractall(path, members, numeric_owner=numeric_owner)

def extract_and_label_folders(tgz_path, output_dir):
    """
    Extracts the tar file and renames class folders in train/val directories.

    Parameters:
    - tgz_path (str): Path to the tar file.
    - output_dir (str): Destination directory for extracted and organized data.
    """
    if not os.path.exists(tgz_path):
        raise FileNotFoundError(f"Tar file not found: {tgz_path}")

    os.makedirs(output_dir, exist_ok=True)

    # Extract contents safely
    print(f"Extracting {tgz_path} to {output_dir}...")
    with tarfile.open(tgz_path, "r") as tar:
        safe_extract(tar, path=output_dir)
    print(f"Extraction completed.")

    # Rename class folders under train and val
    for split in ['train', 'val']:
        split_dir = os.path.join(output_dir, split)
        if not os.path.isdir(split_dir):
            print(f"[Warning] Split directory not found: {split_dir}. Skipping.")
            continue

        for folder_name in os.listdir(split_dir):
            old_path = os.path.join(split_dir, folder_name)
            if os.path.isdir(old_path) and not folder_name.startswith("class_"):
                new_name = f"class_{folder_name}"
                new_path = os.path.join(split_dir, new_name)
                os.rename(old_path, new_path)
                print(f"Renamed: {folder_name} → {new_name}")

# === CONFIGURATION ===
imagenette_tgz_path = "/mnt/abka03/raw_data/winter21_whole.tar"
output_dir = "/mnt/abka03/processed_data/imagenet21"

# === EXECUTION ===
if __name__ == "__main__":
    extract_and_label_folders(imagenette_tgz_path, output_dir)
